# 🇧🇯 Bénin Insights Challenge 2026
**iSHEERO × DataCamp Donates**

> Pipeline : GDELT → Nettoyage → EDA → ML → Insights

---

## Imports & Configuration

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import json, pickle, os
from datetime import datetime, timedelta
from pathlib import Path
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, silhouette_score, ConfusionMatrixDisplay

COLORS = {'primary':'#E91E8C','secondary':'#7B2FBE','accent':'#00D4AA','dark':'#1A1A2E'}
PALETTE= ['#E91E8C','#7B2FBE','#00D4AA','#FF6B35','#4ECDC4','#45B7D1','#96CEB4','#FFEAA7']
plt.rcParams.update({'figure.facecolor':'white','axes.facecolor':'#FAFAFA',
                     'axes.grid':True,'grid.alpha':0.25,'font.size':11,
                     'axes.spines.top':False,'axes.spines.right':False})

BASE      = Path('..')
DATA_PROC = BASE / 'data' / 'processed'
MODELS    = BASE / 'models'
for p in [DATA_PROC, MODELS, BASE/'data'/'raw']: p.mkdir(parents=True, exist_ok=True)
print('✅ Imports OK')

## Chargement des données GDELT

In [ ]:
BIGQUERY_AVAILABLE = False

csv_path = DATA_PROC / 'gdelt_benin_clean.csv'
if csv_path.exists():
    df = pd.read_csv(csv_path, parse_dates=['date'])
    print(f'✅ {len(df):,} événements chargés depuis le CSV')
else:
    print('⚙️  Génération des données prototype...')
    np.random.seed(42); N=8000
    event_codes={'010':'Déclaration publique','020':'Appel action','036':'Critique politique',
        '040':'Consultation diplomatique','050':'Appel coopération','060':'Coopération matérielle',
        '070':'Aide/assistance','080':'Coopération judiciaire','100':'Demande',
        '110':'Rejet/opposition','120':'Menace','130':'Protestation','140':'Refus coopération',
        '145':'Manifestation/révolte','170':'Coercition','180':'Attaque','190':'Lutte non-violente'}
    ek=list(event_codes.keys())
    ew=np.array([.12,.08,.10,.09,.08,.07,.06,.05,.07,.07,.04,.06,.04,.03,.03,.03,.08]); ew/=ew.sum()
    chosen_ev=np.random.choice(ek,N,p=ew)
    cities=['Cotonou','Porto-Novo','Parakou','Abomey-Calavi','Bohicon','Natitingou','Kandi','Lokossa','Ouidah','Abomey','Djougou','Malanville']
    cc={'Cotonou':(6.3654,2.4183),'Porto-Novo':(6.4969,2.6289),'Parakou':(9.337,2.6282),
        'Abomey-Calavi':(6.4484,2.3552),'Bohicon':(7.1824,2.066),'Natitingou':(10.3081,1.3784),
        'Kandi':(11.1342,2.9394),'Lokossa':(6.6392,1.7153),'Ouidah':(6.36,2.0852),
        'Abomey':(7.1856,1.9861),'Djougou':(9.7082,1.6599),'Malanville':(11.8683,3.3836)}
    cw=np.array([.30,.15,.12,.10,.06,.05,.04,.04,.04,.04,.03,.03]); cw/=cw.sum()
    chosen_cities=np.random.choice(cities,N,p=cw)
    base=datetime.now()-timedelta(days=365)
    dates=[base+timedelta(days=int(d)) for d in np.random.choice(365,N)]
    neg=np.isin(chosen_ev,['110','120','130','140','145','170','180'])
    tone=np.where(neg,np.random.normal(-3,2,N),np.random.normal(1,2,N))
    gs={'010':0,'020':0,'036':-2,'040':4,'050':3.4,'060':5,'070':5,'080':5,'100':3,
        '110':-4,'120':-5,'130':-6.5,'140':-7,'145':-7.5,'170':-7,'180':-10,'190':-4}
    gold=np.array([gs.get(e,0)+np.random.normal(0,.5) for e in chosen_ev])
    srcs=['rfi.fr','bbc.com','voaafrique.com','lemonde.fr','fraternite.bj','lanation.bj','beninwebtv.com','afrik.com','reuters.com','apanews.net','allafrica.com','jeune-afrique.com']
    sw=np.array([.02,.10,.08,.09,.10,.08,.07,.06,.08,.06,.08,.08]); sw/=sw.sum()
    qv=np.random.choice([1,2,3,4],N,p=[.25,.25,.25,.25])
    qmap={1:'Coopération verbale',2:'Coopération matérielle',3:'Conflit verbal',4:'Conflit matériel'}
    cameo={'01':'Communication publique','02':'Appels action','03':'Expression intention',
           '04':'Consultation','05':'Coopération','06':'Aide matérielle','07':'Assistance',
           '08':'Accord judiciaire','10':'Demandes','11':'Opposition','12':'Menaces',
           '13':'Protestation','14':'Refus','17':'Coercition','18':'Violence','19':'Lutte'}
    at=['GOV','MIL','OPP','REB','CVL','MED','NGO','BUS','IGO','JUD']
    df=pd.DataFrame({
        'SQLDATE':[d.strftime('%Y%m%d') for d in dates],'date':dates,
        'Actor1Name':np.random.choice(['Gouvernement du Bénin','Patrice Talon','CEDEAO','Armée béninoise','Syndicats','ONU','France','Nigeria','Société civile'],N),
        'Actor1CountryCode':np.random.choice(['BN']*6+['FR','NG','TG','CI'],N),
        'Actor1Type1Code':np.random.choice(at,N),'Actor2Name':np.random.choice(['Population béninoise','Opposition','Journalistes','ONG locales','Communauté internationale'],N),
        'EventCode':chosen_ev,'EventRootCode':[e[:2] for e in chosen_ev],'QuadClass':qv,
        'GoldsteinScale':gold,'NumMentions':np.random.negative_binomial(2,.3,N)+1,
        'NumSources':np.random.negative_binomial(1,.5,N)+1,'NumArticles':np.random.negative_binomial(2,.3,N)+1,
        'AvgTone':tone,'ActionGeo_FullName':chosen_cities,
        'ActionGeo_Lat':[cc[c][0]+np.random.normal(0,.05) for c in chosen_cities],
        'ActionGeo_Long':[cc[c][1]+np.random.normal(0,.05) for c in chosen_cities],
        'SOURCEURL':[f'https://{np.random.choice(srcs,p=sw)}/article/{i}' for i in range(N)],
    })
    df['EventLabel']=df['EventCode'].map(event_codes)
    df['QuadLabel']=df['QuadClass'].map(qmap)
    df['CameoTheme']=df['EventRootCode'].map(cameo).fillna('Autre')
    print(f'✅ {len(df):,} événements générés')

print(f'Shape: {df.shape}'); df.head(3)

## Nettoyage & Feature Engineering

In [ ]:
df['date'] = pd.to_datetime(df['date'], errors='coerce')
df = df.dropna(subset=['date'])
df['year']        = df['date'].dt.year
df['month']       = df['date'].dt.month
df['month_label'] = df['date'].dt.strftime('%b %Y')
df['quarter']     = df['date'].dt.quarter

for c in ['GoldsteinScale','AvgTone','NumMentions','NumSources','NumArticles']:
    df[c] = pd.to_numeric(df[c], errors='coerce').fillna(df[c].median())

df['MediaWeight']  = df['NumMentions'] * df['NumArticles']
df['IsConflict']   = (df['QuadClass'] >= 3).astype(int)
df['IsNegative']   = (df['AvgTone'] < 0).astype(int)
df['ToneCategory'] = pd.cut(df['AvgTone'],bins=[-100,-5,-1,1,5,100],
                     labels=['Très négatif','Négatif','Neutre','Positif','Très positif'])
df['SourceDomain'] = df['SOURCEURL'].str.extract(r'https?://(?:www\.)?([^/]+)')
df['SourceRegion'] = df['SourceDomain'].apply(
    lambda x: 'Afrique' if any(d in str(x) for d in ['bj','afrik','allafrica','apanews','voa','beninweb'])
    else ('Occident' if any(d in str(x) for d in ['bbc','lemonde','reuters','rfi']) else 'Autre'))

df.to_csv(DATA_PROC / 'gdelt_benin_clean.csv', index=False)
print(f'✅ {len(df):,} lignes nettoyées et sauvegardées')
print(f'   Conflits : {df.IsConflict.mean()*100:.1f}%  |  Tone : {df.AvgTone.mean():.2f}')

## Analyse Exploratoire (EDA) — 6 Visualisations

In [ ]:
# VIZ 1 — Timeline
df_wk = df.groupby(df['date'].dt.to_period('W')).agg(nb=('EventCode','count'),tone=('AvgTone','mean')).reset_index()

df_wk['date'] = df_wk['date'].dt.start_time
fig,axes=plt.subplots(2,1,figsize=(14,7),sharex=True)
fig.suptitle('📈 Activité médiatique — Bénin (12 mois)',fontsize=14,fontweight='bold')
axes[0].fill_between(df_wk['date'],df_wk['nb'],alpha=.3,color=COLORS['primary'])
axes[0].plot(df_wk['date'],df_wk['nb'],color=COLORS['primary'],lw=2.5)
axes[0].set_ylabel('Volume hebdomadaire')
clrs=[COLORS['accent'] if t>=0 else COLORS['primary'] for t in df_wk['tone']]
axes[1].bar(df_wk['date'],df_wk['tone'],width=5,color=clrs,alpha=.8)
axes[1].axhline(0,color='black',lw=.8,ls='--')
axes[1].set_ylabel('AvgTone moyen')
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.xticks(rotation=35,ha='right'); plt.tight_layout()
plt.savefig(DATA_PROC/'viz1_timeline.png',dpi=150,bbox_inches='tight'); plt.show()
print('✅ viz1')

In [ ]:
# VIZ 2 — Thèmes & QuadClass
tc=df['CameoTheme'].value_counts().head(10); qc=df['QuadLabel'].value_counts()
fig,axes=plt.subplots(1,2,figsize=(14,6))
fig.suptitle("🏷️ Types d'événements",fontsize=14,fontweight='bold')
axes[0].barh(tc.index[::-1],tc.values[::-1],color=PALETTE[:len(tc)],edgecolor='white',height=.7)
axes[0].set_xlabel("Nombre d'événements"); axes[0].set_title('Top 10 Thèmes CAMEO')
axes[1].pie(qc.values,labels=qc.index,colors=PALETTE[:len(qc)],autopct='%1.1f%%',startangle=90,explode=[0.05]*len(qc))
axes[1].set_title('Répartition QuadClass')
plt.tight_layout(); plt.savefig(DATA_PROC/'viz2_themes.png',dpi=150,bbox_inches='tight'); plt.show()
print('✅ viz2')

In [ ]:
# VIZ 3 — Carte géographique
geo=df.groupby('ActionGeo_FullName').agg(count=('EventCode','count'),tone=('AvgTone','mean'),lat=('ActionGeo_Lat','mean'),lon=('ActionGeo_Long','mean')).reset_index()
fig,ax=plt.subplots(figsize=(9,11)); fig.patch.set_facecolor('#D6EAF8'); ax.set_facecolor('#D6EAF8')
sc=ax.scatter(geo['lon'],geo['lat'],s=geo['count']*1.8,c=geo['tone'],cmap='RdYlGn',vmin=-5,vmax=5,alpha=.85,edgecolors='white',linewidths=1.5,zorder=3)
plt.colorbar(sc,ax=ax,label='Tone moyen',shrink=.65)
for _,row in geo.iterrows():
    ax.annotate(f"{row['ActionGeo_FullName']}\n({int(row['count'])})", (row['lon'],row['lat']),
                textcoords='offset points',xytext=(8,5),fontsize=8.5,fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.2',facecolor='white',alpha=.7,edgecolor='none'))
ax.set_title('🗺️ Événements par ville',fontsize=13,fontweight='bold')
ax.set_xlim(1.0,4.0); ax.set_ylim(5.5,13.0)
plt.tight_layout(); plt.savefig(DATA_PROC/'viz3_carte.png',dpi=150,bbox_inches='tight'); plt.show()
print('✅ viz3')

In [ ]:
# VIZ 4 — Distributions
fig,axes=plt.subplots(1,3,figsize=(15,5))
fig.suptitle('📊 Distributions des scores',fontsize=14,fontweight='bold')
axes[0].hist(df['AvgTone'],bins=50,color=COLORS['primary'],alpha=.8,edgecolor='white')
axes[0].axvline(df['AvgTone'].mean(),color=COLORS['accent'],ls='--',lw=1.5,label=f"Moy={df['AvgTone'].mean():.2f}")
axes[0].set_title('AvgTone'); axes[0].legend(fontsize=9)
axes[1].hist(df['GoldsteinScale'],bins=50,color=COLORS['secondary'],alpha=.8,edgecolor='white')
axes[1].axvline(df['GoldsteinScale'].mean(),color=COLORS['accent'],ls='--',lw=1.5,label=f"Moy={df['GoldsteinScale'].mean():.2f}")
axes[1].set_title('GoldsteinScale'); axes[1].legend(fontsize=9)
tc2=df['ToneCategory'].value_counts()
axes[2].bar(tc2.index,tc2.values,color=PALETTE[:len(tc2)],edgecolor='white')
axes[2].set_title('Catégories sentiment')
plt.setp(axes[2].get_xticklabels(),rotation=25,ha='right',fontsize=9)
plt.tight_layout(); plt.savefig(DATA_PROC/'viz4_distributions.png',dpi=150,bbox_inches='tight'); plt.show()
print('✅ viz4')

In [ ]:
# VIZ 5 — Acteurs & Sources
a1=df['Actor1Name'].value_counts().head(10)
src=df.groupby('SourceDomain').agg(nb=('EventCode','count'),tone=('AvgTone','mean')).sort_values('nb',ascending=False).head(10)
fig,axes=plt.subplots(1,2,figsize=(15,6))
fig.suptitle('👥 Acteurs & 📰 Sources',fontsize=14,fontweight='bold')
axes[0].barh(a1.index[::-1],a1.values[::-1],color=COLORS['secondary'],alpha=.85)
axes[0].set_title('Top 10 acteurs')
clrs2=[COLORS['accent'] if t>=0 else COLORS['primary'] for t in src['tone'].values]
axes[1].barh(src.index[::-1],src['tone'].values[::-1],color=clrs2[::-1],alpha=.85)
axes[1].axvline(0,color='black',lw=.8,ls='--'); axes[1].set_title('Biais tonalité par source')
plt.tight_layout(); plt.savefig(DATA_PROC/'viz5_acteurs.png',dpi=150,bbox_inches='tight'); plt.show()
print('✅ viz5')

In [ ]:
# VIZ 6 — Corrélations
corr_cols=['GoldsteinScale','AvgTone','NumMentions','NumArticles','NumSources','MediaWeight','IsConflict','IsNegative']
corr=df[corr_cols].corr()
fig,ax=plt.subplots(figsize=(10,8))
mask=np.triu(np.ones_like(corr,dtype=bool))
sns.heatmap(corr,mask=mask,cmap=sns.diverging_palette(330,160,as_cmap=True),
            vmin=-1,vmax=1,center=0,annot=True,fmt='.2f',linewidths=.5,ax=ax)
ax.set_title('🔗 Matrice de corrélations',fontsize=14,fontweight='bold')
plt.tight_layout(); plt.savefig(DATA_PROC/'viz6_correlations.png',dpi=150,bbox_inches='tight'); plt.show()
print('✅ viz6 — Toutes les visualisations générées !')

## Machine Learning

In [ ]:
le=LabelEncoder()
df['Actor1Type_enc']=le.fit_transform(df['Actor1Type1Code'].fillna('UNK'))
df['CameoTheme_enc']=le.fit_transform(df['CameoTheme'].fillna('Autre'))
FEAT=['GoldsteinScale','AvgTone','NumMentions','NumArticles','NumSources','MediaWeight','Actor1Type_enc','CameoTheme_enc']
X=df[FEAT].fillna(0); y=df['IsConflict']
scaler=StandardScaler(); X_sc=scaler.fit_transform(X)
print(f'✅ Features: {X.shape} | Conflits: {y.mean()*100:.1f}%')

In [ ]:
# K-Means optimal
sil,inert={},{}
for k in range(2,9):
    km=KMeans(n_clusters=k,random_state=42,n_init=10)
    lbl=km.fit_predict(X_sc)
    sil[k]=silhouette_score(X_sc,lbl,sample_size=2000)
    inert[k]=km.inertia_
    print(f'k={k}: silhouette={sil[k]:.4f}')
best_k=max(sil,key=sil.get)
print(f'\n✅ k optimal = {best_k}')

fig,axes=plt.subplots(1,2,figsize=(12,4))
axes[0].plot(list(sil.keys()),list(sil.values()),'o-',color=COLORS['primary'],lw=2.5,ms=8)
axes[0].axvline(best_k,ls='--',color=COLORS['accent'],label=f'k={best_k}')
axes[0].set_xlabel('k'); axes[0].set_ylabel('Silhouette'); axes[0].legend()
axes[1].plot(list(inert.keys()),list(inert.values()),'s-',color=COLORS['secondary'],lw=2.5,ms=8)
axes[1].axvline(best_k,ls='--',color=COLORS['accent'])
axes[1].set_xlabel('k'); axes[1].set_ylabel('Inertie')
plt.tight_layout(); plt.show()

In [ ]:
# K-Means final + PCA
kmeans=KMeans(n_clusters=best_k,random_state=42,n_init=10)
df['Cluster']=kmeans.fit_predict(X_sc)
pca=PCA(n_components=2,random_state=42)
Xpca=pca.fit_transform(X_sc)
df['PCA1'],df['PCA2']=Xpca[:,0],Xpca[:,1]

fig,ax=plt.subplots(figsize=(10,7))
for i,c in enumerate(sorted(df['Cluster'].unique())):
    mask=df['Cluster']==c
    ax.scatter(df.loc[mask,'PCA1'],df.loc[mask,'PCA2'],s=15,alpha=.45,color=PALETTE[i],label=f'Cluster {c}')
centers_2d=pca.transform(kmeans.cluster_centers_)
ax.scatter(centers_2d[:,0],centers_2d[:,1],s=200,marker='X',color='black',edgecolors='white',lw=1.5,zorder=5,label='Centroïdes')
ax.set_title(f'K-Means k={best_k} — PCA 2D'); ax.legend()
plt.tight_layout(); plt.savefig(DATA_PROC/'viz_clusters.png',dpi=150,bbox_inches='tight'); plt.show()

profile=df.groupby('Cluster').agg(
    Événements=('EventCode','count'),Tone_moyen=('AvgTone','mean'),
    Goldstein_moy=('GoldsteinScale','mean'),Thème_top=('CameoTheme',lambda x:x.mode()[0]),
    Ville_top=('ActionGeo_FullName',lambda x:x.mode()[0]),
    Conflits_pct=('IsConflict',lambda x:f'{x.mean()*100:.0f}%')).round(2)
print(profile.to_string())

In [ ]:
# Random Forest
Xtr,Xte,ytr,yte=train_test_split(X_sc,y,test_size=0.2,random_state=42,stratify=y)
rf=RandomForestClassifier(n_estimators=200,max_depth=10,min_samples_leaf=5,
                           random_state=42,n_jobs=-1,class_weight='balanced')
rf.fit(Xtr,ytr); y_pred=rf.predict(Xte)
report=classification_report(yte,y_pred,target_names=['Coopération','Conflit'],output_dict=True)
cm=confusion_matrix(yte,y_pred)
print(classification_report(yte,y_pred,target_names=['Coopération','Conflit']))

fig,axes=plt.subplots(1,2,figsize=(13,5))
fig.suptitle('🌲 Random Forest',fontsize=14,fontweight='bold')
ConfusionMatrixDisplay(cm,display_labels=['Coopération','Conflit']).plot(ax=axes[0],cmap='RdYlGn',colorbar=False)
imp=pd.Series(rf.feature_importances_,index=FEAT).sort_values()
axes[1].barh(imp.index,imp.values,color=COLORS['secondary'],alpha=.85)
axes[1].set_title('Importance des variables')
plt.tight_layout(); plt.savefig(DATA_PROC/'viz_rf.png',dpi=150,bbox_inches='tight'); plt.show()

In [ ]:
# Sauvegarde modèles
pickle.dump({'scaler':scaler,'kmeans':kmeans,'pca':pca,'rf':rf,'feat':FEAT,
             'best_k':best_k,'sil_scores':sil,'report':report,'profile':profile.to_dict()},
            open(MODELS/'models_bundle.pkl','wb'))
print('✅ models/models_bundle.pkl sauvegardé')

## Les 5 Insights Clés

In [ ]:
df_wk2=df.groupby(df['date'].dt.to_period('W')).agg(nb=('EventCode','count')).reset_index()
insights={
    '1':{'icon':'📈','title':"L'activité médiatique suit des cycles clairs",
         'body':f"{len(df):,} événements sur 12 mois. Pic hebdo: {int(df_wk2['nb'].max())}."},
    '2':{'icon':'⚠️','title':f"{df['IsConflict'].mean()*100:.0f}% des événements sont conflictuels",
         'body':f"Seulement {(1-df['IsConflict'].mean())*100:.0f}% relève de la coopération."},
    '3':{'icon':'🏙️','title':"Cotonou concentre la quasi-totalité de la visibilité",
         'body':f"{(df['ActionGeo_FullName']=='Cotonou').mean()*100:.0f}% des événements à Cotonou."},
    '4':{'icon':'📰','title':"Tonalité médiatique légèrement négative",
         'body':f"Tone moyen: {df['AvgTone'].mean():.2f}. {df['IsNegative'].mean()*100:.0f}% négatifs."},
    '5':{'icon':'🤝','title':"Les accords diplomatiques sont sous-couverts",
         'body':f"Cluster {profile['Goldstein_moy'].idxmax()}: Goldstein={profile['Goldstein_moy'].max():.1f} mais faible couverture."},
}
json.dump(insights,open(DATA_PROC/'insights.json','w',encoding='utf-8'),ensure_ascii=False,indent=2)
print('💡 LES 5 INSIGHTS')
for k,ins in insights.items():
    print(f"\n#{k} {ins['icon']} {ins['title']}")
    print(f"   → {ins['body']}")

In [ ]:
print('='*60)
print('  ✅  PIPELINE COMPLET — BÉNIN INSIGHTS 2026')
print('='*60)
print(f'  Dataset     : {len(df):,} lignes × {df.shape[1]} colonnes')
print(f'  Conflits    : {df.IsConflict.mean()*100:.1f}%')
print(f'  Tone moyen  : {df.AvgTone.mean():.3f}')
print(f'  K-Means k   : {best_k}')
print(f'  RF Accuracy : {report["accuracy"]:.2%}')
print('\n  streamlit run dashboard/app.py')
print('='*60)